# OpenET monthly evapotranspiration over the Ogallala aquifer at 30 m: ensemble and six models, one CF file per tile and year

OpenET monthly totals (release 2.1 by default, `OPENET_VERSION`), October 1999 → the latest month OpenET has, at **OpenET's own 30 m
pixels**. The stored integers are downloaded from Earth Engine to Colab's scratch disk and written to the
files as they are. Nothing is computed on Earth Engine, so almost none of the project's compute quota is used.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/OPENET_v2_1_monthly_30m_tiles/<year>/<tile>/OPENET_v2_1_monthly_30m_tiles_<tile>_<year>_Ogallala.nc`: a folder per year, in it a folder per tile, in it that tile's file of the year |
| Months | `START_MONTH` to `STOP_MONTH` (Cell 2), e.g. `"2005-01"` to `"2005-12"` |
| One record | one month: the monthly total of each model and of the ensemble, in whole millimetres |
| Grid | OpenET's own 30 m grid of each UTM zone, cut into tiles of 5120 × 5120 pixels. It is the grid Landsat uses, and the tiles are those of the Landsat index files of this project, so the two line up pixel for pixel. **Nothing is resampled** |
| Pixels kept | pixels whose centre lies inside the aquifer polygon |
| Variables | `et_ensemble_mad` (the OpenET ensemble value), `et_ensemble_mad_min`, `et_ensemble_mad_max`, `et_ensemble_mad_count`, `et_ensemble_mad_index`, `et_ensemble_sam`; for each of DisALEXI, eeMETRIC, geeSEBAL, PT-JPL, SIMS and SSEBop: `et_<model>` and `count_<model>` |

**How to run**

1. *Run all* with `TEST_MODE = True`: 2 tiles, June and July of 2022, into `.../_test/`, every check
   switched on, and a projection of runtime, download volume and Drive size for the months from
   `START_MONTH` to `STOP_MONTH`. No GPU is needed.
2. Set `TEST_MODE = False`, set `START_MONTH` and `STOP_MONTH`, and *Run all*. Whenever the runtime dies or
   you stop it, run all again. `ONLY_TILES` lets several Colab sessions work on different tiles; never
   run two sessions on the same tile.

**Choosing the months.** `START_MONTH = "2005-01"`, `STOP_MONTH = "2005-12"` exports those twelve months
and nothing else; `STOP_MONTH = None` goes on to the latest month OpenET has. Ranges can be run in any
order and may overlap. A later run adds its months to the yearly files that are on Drive; a month that
is already there is never fetched again, and its stored values are never changed.

**What the values are**

* **OpenET's numbers, untouched.** Every variable is one band of an OpenET monthly image, copied as the
  integer OpenET stores (its scale factor is 1, the unit is mm per month). No scaling, no gap filling, no
  screening and no resampling happens here.
* **The ensemble.** `et_ensemble_mad` is OpenET's ensemble value: the mean of the models left after outliers
  were removed by the median absolute deviation. `_min` and `_max` are the smallest and largest of those
  models, `_count` how many were left, `et_ensemble_sam` the plain mean of all models.
* **The models.** `et_<model>` is each model's monthly total. `count_<model>` is the number of cloud-free
  Landsat observations that total rests on; a month with a count of 0 was filled by OpenET from
  neighbouring dates. DisALEXI starts in 2001, so its variables are empty before that.
* **Missing data.** Where OpenET has no value the files hold −32768, which xarray and GDAL read as
  missing; the same value marks pixels outside the aquifer polygon. A stored 0 is a real 0 mm.
  Release 2.0 writes −32768 itself. Release 2.1 does not: there a pixel without data and a pixel with
  0 mm both download as 0, so the notebook reads OpenET's mask from Earth Engine as well (one small
  computed image per block, one bit per collection) and marks exactly the pixels the mask excludes.
  This mask read is the only part that uses Earth Engine compute quota; the rehearsal reports how
  many such reads it made and how long they took.
* **What a month's value rests on.** OpenET computes ET on the days Landsat saw the ground and fills the
  days between by interpolation, then sums the month. The monthly totals here are those sums.

**How it works** (per tile and year: fetch a month → check → write → next month → check the file → Drive)

* The image list comes from Earth Engine's catalogue and is cached on Drive. OpenET stores a month as one
  image per UTM zone and latitude band; the images of a zone lie side by side and do not overlap. Each
  tile's frame (the tile cut to the aquifer) is stored in `frames.json` and never changed afterwards.
* For a tile and month, the part of every image that lies on the tile is fetched in blocks as the stored
  integers, up to 32 requests at a time. Earth Engine refuses a request above 48 MiB; a refused request
  is fetched in halves instead of failing. Every block is checked when it arrives and compared with a
  checksum when it is read back from scratch.
* Before any bulk work, a small window of one image per collection is compared with Earth Engine's own
  reading of the same pixels; they must be identical.
* While one month is being written, the next one is already downloading.
* A yearly file holds the months made so far. A later run that wants more months of that year fetches
  the file from Drive and adds them. If a new month belongs before one that is there, a new file is
  written and the months on Drive are copied into it as they are stored.

**Stopping and resuming**

* The unit of work is one tile and one year. Its file is built on scratch, read back and audited, and
  only then copied to Drive under its final name, with a checksum comparison. Nothing on Drive is ever
  half written.
* A runtime that dies loses the file it was working on and nothing else. The next run starts with the
  first file that is missing or lacks a month of the range.
* If a file on Drive cannot be read, the run stops and changes nothing. It never replaces a file on its own.

**What to know before using the data**

* The aquifer lies in UTM zones 13 and 14 (and a corner of 15). Tiles of different zones are on different
  grids and are not merged; along a zone boundary both zones hold the same ground.
* OpenET publishes new months with a delay and can reissue a collection. A month that is already in a
  file is not fetched again, so a reissue is not picked up unless the file is removed from Drive first.
  A month is only written once every chosen collection has it, so no month is left with a model missing.
* Release 2.1 is a reprocessing of 2.0 (more cloud screening, updated weather and land-cover inputs,
  model updates). OpenET says values can differ noticeably from 2.0 in places; do not mix the two.

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# earthengine-api, numpy, geopandas, shapely and pyproj are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q netCDF4 rasterio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE     = True       # True -> short rehearsal into .../_test/ ; False -> the months from START_MONTH to STOP_MONTH
TEST_TILES    = 2          # tiles in the rehearsal (one mostly inside the aquifer, one on its edge)
TEST_YEAR     = 2022       # year of the rehearsal
TEST_MONTHS   = [6, 7]     # months of that year in the rehearsal
TEST_FRAME_PX = None       # None -> whole tiles; or e.g. 2048 to rehearse on a 2048 x 2048 pixel square of each

# ---- which months are processed and exported (not used by the rehearsal) ----
START_MONTH = "1999-10"    # first month, written "YYYY-MM" (OpenET's monthly record starts in October 1999)
STOP_MONTH  = None         # last month, written "YYYY-MM" (it is included); None -> the latest month OpenET has.
                           # Any range can be run in any order: a later run with another range adds its months
                           # to the yearly files that are already on Drive and never fetches a month twice

PROJECT_ID = "msugw-503806"                                    # Earth Engine cloud project
EE_URL     = "https://earthengine-highvolume.googleapis.com"   # endpoint meant for many parallel requests
# OpenET monthly collections at 30 m: the ensemble and the six models it is made of
OPENET_VERSION = "v2_1"    # which release to fetch. "v2_1": the current one, a reprocessing of v2_0 that fixes known
                           # problems and goes on being extended; "v2_0": the earlier one, which ends in December
                           # 2024. Change this one setting to get the other release; each release goes to its own
                           # output folder (OPENET_v2_1_... / OPENET_v2_0_...), so the two are never mixed
MODELS     = ["ensemble", "disalexi", "eemetric", "geesebal", "ptjpl", "sims", "ssebop"]
TILE_PX    = 5120          # edge of an output tile in 30 m pixels (153.6 km): the same tiles as the Landsat index
                           # files, so the two line up pixel for pixel. Do not change it once files exist
ONLY_TILES = None          # None -> every tile; or e.g. ["z14e02n29", "z14e02n28"] (to split the work over sessions)
SKIP_IMAGES = []           # image ids to treat as permanently unavailable (see the stall message)

# ---- download ----
BLOCK_PX           = 2896  # a download block holds at most BLOCK_PX x BLOCK_PX pixels. Earth Engine counts 3 bytes
                           # per band value and refuses a request above 48 MiB; the limit for each image's number
                           # of bands is applied by itself, and a refused request is fetched in halves
N_DOWNLOAD_THREADS = 32    # blocks downloaded at a time (Earth Engine allows 40 concurrent requests)
REQUEST_DEADLINE_S = 180   # give up on a single request after this long (it is then retried)
STRIP_ROWS         = 512   # rows placed and written at a time (also the chunk height of the files)

COMPRESS_LEVEL = 2         # zlib level of the .nc files (lossless)
CHUNK_YX       = 512       # HDF5 chunk edge in y and x

MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
DRIVE_READ_TRIES  = 4      # attempts at reading a file's state from Drive before the run stops. The run never
                           # redoes or replaces a file because Drive failed to open it
MAX_RETRIES       = 4      # extra attempts per block
STALL_MINUTES     = 20     # stop cleanly after this long with no progress at all
READBACK_MONTHS   = 1      # months per file re-read from the closed file and compared with what was written
                           # (every month in TEST_MODE)
MASK_BLOCK_PX     = 2896   # edge of a block of the computed mask read (release 2.1, see the first cell)
CHECK_PX          = 64     # edge of the window compared with Earth Engine's own reading of the same pixels

MANIFEST_MAX_AGE_DAYS = 7  # refresh the cached image list when older than this
REBUILD_MANIFEST      = False
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, Earth Engine sign-in, the variables ═══════════════
import re, gc, io, sys, json, gzip, glob, time, math, zlib, shutil, hashlib, logging, threading, warnings
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import pyproj
import shapely
import rasterio.features
from affine import Affine
import ee

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a no-op there.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("openet")
for _n in ("urllib3.connectionpool", "googleapiclient", "google_auth_httplib2", "rasterio", "pyogrio"):
    logging.getLogger(_n).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
if OPENET_VERSION not in ("v2_0", "v2_1"):
    raise ValueError(f'OPENET_VERSION = {OPENET_VERSION!r}: write "v2_1" or "v2_0".')
COLLECTION  = "projects/openet/assets/{model}/conus/gridmet/monthly/" + OPENET_VERSION
PRODUCT_TAG = f"OPENET_{OPENET_VERSION}_monthly_30m_tiles"
DRIVE_ROOT  = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT   = f"{DRIVE_ROOT}/{PRODUCT_TAG}" + ("/_test" if TEST_MODE else "")   # the rehearsal never touches real outputs
SCRATCH     = "/content/openet_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR     = f"{SCRATCH}/raw"                      # downloaded pixels of one or two months, then deleted
NC_DIR      = f"{SCRATCH}/nc"                       # the yearly file being written
MASK_DIR    = f"{SCRATCH}/masks"                    # aquifer mask of each tile
DISK_PATH   = "/content"                            # same filesystem as scratch AND Drive's local upload cache

MANIFEST_NAME = f"manifest_openet_{OPENET_VERSION}.json.gz"
FRAMES_NAME   = "frames.json"                       # the fixed 30 m frame of every tile (made once, never changed)
LEDGER_NAME   = "files_done.jsonl"                  # one line per yearly file that is on Drive
LOG_FAILED    = "failed_images.jsonl"
LOG_SKIPPED   = "skipped_images.jsonl"
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, MASK_DIR):
    os.makedirs(_d, exist_ok=True)

PIXEL = 30.0                                        # OpenET's own pixel size (m); nothing is resampled
TIME_UNITS, TIME_CALENDAR = "days since 1970-01-01 00:00:00", "proleptic_gregorian"
EPOCH = datetime(1970, 1, 1)
FILL  = -32768                                      # OpenET's own "no data" value of its 16-bit bands
NO_COUNT = 255                                      # fill of the 8-bit count variables

# ---------- the variables: every band of every chosen collection, stored as OpenET stores it ----------
MODEL_NAME = dict(ensemble="OpenET ensemble", disalexi="ALEXI/DisALEXI", eemetric="eeMETRIC", geesebal="geeSEBAL",
                  ptjpl="PT-JPL", sims="SIMS", ssebop="SSEBop")
ENSEMBLE_BANDS = dict(
    et_ensemble_mad="ensemble ET: mean of the models left after outliers were removed by median absolute deviation",
    et_ensemble_mad_min="smallest model value among the models left after outlier removal",
    et_ensemble_mad_max="largest model value among the models left after outlier removal",
    et_ensemble_mad_count="number of models left after outlier removal",
    et_ensemble_mad_index="bit mask of the models used in the ensemble value (as defined by OpenET)",
    et_ensemble_sam="simple arithmetic mean of all models")
BANDS = {m: (list(ENSEMBLE_BANDS) if m == "ensemble" else ["et", "count"]) for m in MODELS}


def var_name(model, band):
    return band if model == "ensemble" else f"{band}_{model}"


VARS = [var_name(m, b) for m in MODELS for b in BANDS[m]]          # in file order
IS_COUNT = {v: v.startswith("count_") for v in VARS}               # 8-bit in OpenET; everything else is 16-bit
VAR_FILL = {v: (NO_COUNT if IS_COUNT[v] else FILL) for v in VARS}
VAR_TYPE = {v: ("u1" if IS_COUNT[v] else "i2") for v in VARS}
MAIN_VAR = VARS[0]


class PipelineStop(RuntimeError):
    """Clean stop: everything on Drive is safe, re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


class SceneUnavailable(Exception):
    """Earth Engine says the image does not exist. Nothing else may be mistaken for this: an image
    reported unavailable is left out of the record for good."""


class RunStopping(Exception):
    """Raised inside a download thread when the run is being stopped."""


class RequestTooLarge(Exception):
    """Earth Engine refused a pixel request for its size. Trying again cannot help; the rectangle
    is fetched as two halves instead."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"


def parse_iso(s):
    s = s.rstrip("Z")
    return datetime.strptime(s, "%Y-%m-%dT%H:%M:%S.%f" if "." in s else "%Y-%m-%dT%H:%M:%S")


def ee_login():
    """Stored credentials first (an unattended restart needs no clicks); otherwise the sign-in flow."""
    try:
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID, opt_url=EE_URL)
    ee.data.setDeadline(REQUEST_DEADLINE_S * 1000)
    if ee.Number(1).add(1).getInfo() != 2:
        raise PipelineStop("Earth Engine did not answer a trivial request.")
    log.info("Earth Engine ready (project %s, high-volume endpoint)", PROJECT_ID)


log.info("%d collections, %d variables: %s", len(MODELS), len(VARS), " ".join(VARS))

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: aquifer polygon, months, image list, the tile grid of each UTM zone, block plan ═══════════════
_ROI_GDF = gpd.read_file(SHP_PATH)
if _ROI_GDF.crs is None:
    raise ValueError(f"{SHP_PATH} has no CRS")
_POLY = {}


def roi_polygon(crs):
    """The aquifer polygon in a given CRS (one copy per CRS). The shapefile's parts are merged once,
    in its own CRS; the merged outline is then reprojected, which is quick."""
    if None not in _POLY:
        g = _ROI_GDF.geometry
        u = g.union_all() if hasattr(g, "union_all") else g.unary_union
        _POLY[None] = u if u.is_valid else u.buffer(0)
    if crs not in _POLY:
        u = gpd.GeoSeries([_POLY[None]], crs=_ROI_GDF.crs).to_crs(crs).iloc[0]
        _POLY[crs] = u if u.is_valid else u.buffer(0)
    return _POLY[crs]


log.info("Aquifer polygon: %.0f km2", roi_polygon("EPSG:5070").area / 1e6)


def with_retries(fn, what, tries=None, stop=None):
    """Run an Earth Engine request; retry with growing pauses. A missing asset is not retried.
    `stop` (a threading.Event) ends the waiting at once when the run is being stopped."""
    n = 1 + (MAX_RETRIES if tries is None else tries)
    last = None
    for attempt in range(n):
        if stop is not None and stop.is_set():
            raise RunStopping(what)
        try:
            return fn()
        except Exception as e:
            last = e
            msg = str(e)
            low = msg.lower()
            if "request size" in low and "must be less than" in low:
                raise RequestTooLarge(msg[:200])
            if "asset" in low and ("not found" in low or "does not exist" in low):
                raise SceneUnavailable(f"{what}: {msg[:200]}")
            if attempt + 1 < n:
                quota = "quota" in low or "429" in msg or "too many" in low
                pause = min((20 if quota else 2) * 2 ** attempt, 300)
                if stop is None:
                    time.sleep(pause)
                elif stop.wait(pause):
                    raise RunStopping(what)
    raise IOError(f"{what}: {type(last).__name__}: {str(last)[:300]}")


# ---------- which months ----------
def parse_month(text, what):
    """ "YYYY-MM" -> (year, month)."""
    m = re.fullmatch(r"(\d{4})-(\d{1,2})", str(text).strip())
    if not m or not 1 <= int(m.group(2)) <= 12:
        raise ValueError(f'{what} = {text!r} is not a month. Write it as "YYYY-MM", for example "2005-01".')
    return int(m.group(1)), int(m.group(2))


FIRST_MONTH = parse_month(START_MONTH, "START_MONTH")
LAST_MONTH  = parse_month(STOP_MONTH, "STOP_MONTH") if STOP_MONTH is not None else None
if LAST_MONTH is not None and LAST_MONTH < FIRST_MONTH:
    raise ValueError(f"STOP_MONTH ({STOP_MONTH}) is before START_MONTH ({START_MONTH}).")


def month_bounds(year, month):
    start = datetime(year, month, 1)
    return start, datetime(year + (month == 12), month % 12 + 1, 1)


def ym(year, month):
    return f"{year}-{month:02d}"


def span(months):
    """[1, 2, 3, 7, 8] -> '1-3, 7-8'."""
    out, first = [], None
    for i, m in enumerate(months):
        first = m if first is None else first
        if i + 1 == len(months) or months[i + 1] != m + 1:
            out.append(str(m) if first == m else f"{first}-{m}")
            first = None
    return ", ".join(out)


AVAILABLE = set()        # (year, month) for which OpenET has an image over the aquifer; filled from the image list


def range_months():
    """Every (year, month) from START_MONTH to STOP_MONTH for which OpenET has data, in order."""
    return sorted(k for k in AVAILABLE if k >= FIRST_MONTH and (LAST_MONTH is None or k <= LAST_MONTH))


_JOB = {}


def job_months(refresh=False):
    """The (year, month) list of this run: in the rehearsal the chosen months, otherwise
    range_months(). Worked out once per run, so the list cannot change while files are written."""
    if refresh or "months" not in _JOB:
        months = [(TEST_YEAR, m) for m in sorted(set(TEST_MONTHS)) if (TEST_YEAR, m) in AVAILABLE] if TEST_MODE             else range_months()
        by_year = {}
        for y, m in months:
            by_year.setdefault(y, []).append(m)
        _JOB.update(months=months, by_year=by_year)
    return _JOB["months"]


def year_list():
    job_months()
    return sorted(_JOB["by_year"])


def wanted_months(year):
    """Months of a year that this run is to have on Drive."""
    job_months()
    return _JOB["by_year"].get(year, [])


# ---------- image list (cached on Drive) ----------
# OpenET stores a month as one image per UTM zone and latitude band ("14s"); the images of a zone lie
# side by side on one 30 m grid and do not overlap.
def _image_record(model, im):
    """One catalogue entry -> what the pipeline needs. Returns (record, None) or (None, reason)."""
    bands = {b["id"]: b for b in im.get("bands", [])}
    need = BANDS[model]
    if any(k not in bands for k in need):
        return None, f"bands missing: {[k for k in need if k not in bands]}"
    g = bands[need[0]]["grid"]
    if any(bands[k]["grid"] != g for k in need):
        return None, "bands are not on one common grid"
    at = g["affineTransform"]
    if at.get("shearX", 0) or at.get("shearY", 0):
        return None, "rotated grid"
    sid = im["id"].split("/")[-1]
    return dict(id=f"{model}/{sid}", name=im["name"], model=model, tag=sid.split("_")[0], t=im["startTime"],
                bands=need, signed=bands[need[0]]["dataType"].get("range", {}).get("min", 0) < 0, crs=g["crsCode"], tx=float(at["translateX"]), ty=float(at["translateY"]),
                sx=float(at["scaleX"]), sy=float(at["scaleY"]), w=int(g["dimensions"]["width"]),
                h=int(g["dimensions"]["height"])), None


def _list_collection(model):
    """Every image of a collection whose footprint reaches the aquifer."""
    parent = COLLECTION.format(model=model)
    if not parent.startswith("projects/"):
        parent = f"projects/earthengine-public/assets/{parent}"
    recs, odd, token, reach = [], [], None, {}
    while True:
        params = dict(parent=parent, pageSize=1000, view="FULL")
        if token:
            params["pageToken"] = token
        page = with_retries(lambda: ee.data.listImages(params), f"listing {model}")
        for im in page.get("images", []):
            rec, why = _image_record(model, im)
            if rec is None:
                odd.append(dict(image=im.get("id", "?"), reason=why))
                continue
            key = (rec["crs"], rec["tx"], rec["ty"], rec["w"], rec["h"])
            if key not in reach:
                box = shapely.geometry.box(rec["tx"], rec["ty"] + rec["h"] * rec["sy"], rec["tx"] + rec["w"] * rec["sx"], rec["ty"])
                reach[key] = roi_polygon(rec["crs"]).intersects(box)
            if reach[key]:
                recs.append(rec)
        token = page.get("nextPageToken")
        if not token:
            return recs, odd


def get_manifest():
    """Every image of the chosen collections that reaches the aquifer, with its grid."""
    sig = json.dumps(dict(models=MODELS, collection=COLLECTION), sort_keys=True)
    path = os.path.join(DRIVE_OUT, MANIFEST_NAME)
    if os.path.exists(path) and not REBUILD_MANIFEST:
        try:
            with gzip.open(path, "rt") as f:
                m = json.load(f)
            age = (utcnow() - parse_iso(m["created"])).total_seconds() / 86400
            if m.get("signature") == sig and age <= MANIFEST_MAX_AGE_DAYS:
                log.info("Image list from Drive: %d images, %.1f days old", len(m["images"]), age)
                return m["images"]
        except Exception as e:
            log.warning("Cached image list unreadable (%s) - listing again", e)
    t0 = time.time()
    recs, odd = [], []
    with ThreadPoolExecutor(min(8, len(MODELS))) as ex:
        for r, o in ex.map(_list_collection, MODELS):
            recs += r
            odd += o
    recs.sort(key=lambda r: (r["t"], r["id"]))
    log_events(LOG_SKIPPED, [dict(o, stage="image list") for o in odd])
    local = os.path.join(NC_DIR, MANIFEST_NAME)
    with gzip.open(local, "wt") as f:
        json.dump(dict(created=iso(utcnow()), signature=sig, images=recs), f)
    robust_drive_copy(local, path)
    os.remove(local)
    log.info("Image list: %d images over the aquifer (%s), %d unusable entries, %.0f s", len(recs),
             ", ".join(f"{m} {sum(1 for r in recs if r['model'] == m)}" for m in MODELS), len(odd), time.time() - t0)
    return recs


# ---------- the tile grid of each UTM zone ----------
# OpenET's images of a UTM zone lie on one 30 m lattice, the same one Landsat uses. Each zone is cut into
# square tiles of that lattice (the tiles of the Landsat index files), and a tile takes the pixels of
# every image of its zone that covers it, placed by whole-pixel shifts. A tile's frame is the tile cut to
# the aquifer; it depends on geometry only, is stored on Drive and never changes.
TILE_M = TILE_PX * PIXEL


def tile_key(crs, i, j):
    return f"z{int(crs.split(':')[1]) % 100:02d}e{i:02d}n{j:02d}"


def _mask_path(key):
    return os.path.join(MASK_DIR, f"{key}.npy")


def _rasterize(F):
    return rasterio.features.geometry_mask([roi_polygon(F["crs"])], out_shape=(F["h"], F["w"]), invert=True,
                                           all_touched=False,          # pixel centre inside the polygon
                                           transform=Affine(PIXEL, 0.0, F["x0"], 0.0, -PIXEL, F["y0"]))


def _save_mask(key, mask):
    tmp = _mask_path(key) + ".part"
    with open(tmp, "wb") as f:
        np.save(f, np.packbits(mask))
    os.replace(tmp, _mask_path(key))


def frame_mask(key, F):
    """True where the pixel centre lies inside the aquifer polygon (kept on scratch, 1 bit per pixel)."""
    if os.path.exists(_mask_path(key)):
        bits = np.load(_mask_path(key))
        if bits.size * 8 >= F["h"] * F["w"]:
            mask = np.unpackbits(bits)[:F["h"] * F["w"]].reshape(F["h"], F["w"]).astype(bool)
            if int(mask.sum()) == F["n_roi"]:
                return mask
    mask = _rasterize(F)
    if int(mask.sum()) != F["n_roi"]:
        raise DataIntegrityError(f"{key}: the aquifer mask has {int(mask.sum())} pixels, its frame says {F['n_roi']}. "
                                 "The shapefile differs from the one the frames were made with.")
    _save_mask(key, mask)
    return mask


def on_lattice(r, ref):
    return (r["crs"] == ref["crs"] and r["sx"] == PIXEL and r["sy"] == -PIXEL
            and (r["tx"] - ref["tx"]) % PIXEL == 0 and (r["ty"] - ref["ty"]) % PIXEL == 0)


def make_frame(key, crs, ox, oy, i, j):
    """Frame of tile (i, j) of a zone whose lattice has its pixel edges at ox, oy (mod 30 m): the
    tile cut to the rectangle that bounds its aquifer pixels. None when it holds no aquifer pixel."""
    left, top = ox + i * TILE_M, oy + (j + 1) * TILE_M
    box = shapely.geometry.box(left, top - TILE_M, left + TILE_M, top)
    if not roi_polygon(crs).intersects(box):
        return None
    F = dict(crs=crs, x0=left, y0=top, w=TILE_PX, h=TILE_PX)
    mask = _rasterize(F)
    if not mask.any():
        return None
    rows, cols = np.flatnonzero(mask.any(axis=1)), np.flatnonzero(mask.any(axis=0))
    r0, r1, c0, c1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
    if TEST_MODE and TEST_FRAME_PX:                       # a small square around the aquifer pixel nearest the centre
        rr, cc = np.nonzero(mask[r0:r1:8, c0:c1:8])
        if not rr.size:
            return None
        k = int(np.argmin((rr * 8 - (r1 - r0) / 2) ** 2 + (cc * 8 - (c1 - c0) / 2) ** 2))
        ra = max(r0, r0 + int(rr[k]) * 8 - TEST_FRAME_PX // 2)
        ca = max(c0, c0 + int(cc[k]) * 8 - TEST_FRAME_PX // 2)
        r0, r1, c0, c1 = ra, min(r1, ra + TEST_FRAME_PX), ca, min(c1, ca + TEST_FRAME_PX)
    sub = np.ascontiguousarray(mask[r0:r1, c0:c1])
    F = dict(crs=crs, x0=left + c0 * PIXEL, y0=top - r0 * PIXEL, w=c1 - c0, h=r1 - r0, n_roi=int(sub.sum()))
    _save_mask(key, sub)
    return F


def get_frames(cand):
    """Frames of the candidate tiles {key: (crs, ox, oy, i, j)}: read from Drive, and made (then
    stored) for tiles that have none yet."""
    path = os.path.join(DRIVE_OUT, FRAMES_NAME)
    tag = dict(pixel=PIXEL, tile_px=TILE_PX, test_frame_px=TEST_FRAME_PX if TEST_MODE else None)
    stored = {}
    if os.path.exists(path):
        with open(path) as f:
            doc = json.load(f)
        if doc.get("made_with") != tag:
            if not TEST_MODE:
                raise PipelineStop(f"{FRAMES_NAME} on Drive was made with other settings ({doc.get('made_with')}); "
                                   "this run would not fit the files next to it. Use an empty output folder.")
            doc = {}
        stored = doc.get("frames", {})
    new, t0 = 0, time.time()
    for key in sorted(cand):
        if key not in stored:
            stored[key] = make_frame(key, *cand[key])       # None is stored too: "no aquifer pixel here"
            new += 1
    if new:
        local = os.path.join(NC_DIR, FRAMES_NAME)
        with open(local, "w") as f:
            json.dump(dict(made_with=tag, created=iso(utcnow()), frames=stored), f, indent=1)
        robust_drive_copy(local, path)
        os.remove(local)
        log.info("Frames made for %d tiles in %.0f s and stored on Drive", new, time.time() - t0)
    return {k: stored[k] for k in cand if stored.get(k)}


# ---------- tiles OpenET has nothing for ----------
# OpenET cuts its images at the edges of each UTM zone, while the tile grid of a zone reaches a little
# further (as far as Landsat scenes of that zone do). Such a tile would be all "no data": the same
# ground is held by the tiles of the neighbouring zone. Whether a tile has data is found once, by reading
# every 256th row and column of the tile from one month of the ensemble at full resolution, and stored
# on Drive. A tile is left out only when every one of those lines is "no data".
COVERAGE_NAME, LINE_STEP = "coverage.json", 256


def _has_data(P):
    F = P["F"]
    month = max(k for k, v in P["months"].items() if any(r["model"] == MODELS[0] for r in v))
    for r in P["months"][month]:
        if r["model"] != MODELS[0]:
            continue
        S, band = scene_window(r, F), r["bands"][:1]
        for k in range(LINE_STEP // 2, max(S["h"], S["w"]), LINE_STEP):
            for row0, col0, h, w in ((k, 0, 1, S["w"]), (0, k, S["h"], 1)):
                if row0 < S["h"] and col0 < S["w"]:
                    if valid_pixels(r, band[0], S["sr0"] + row0, S["sc0"] + col0, h, w).any():
                        return True, ym(*month)
    return False, ym(*month)


def with_data(tiles):
    """The tiles for which OpenET holds any value (see above); the answer per tile is kept on Drive."""
    path = os.path.join(DRIVE_OUT, COVERAGE_NAME)
    tag = dict(tile_px=TILE_PX, test_frame_px=TEST_FRAME_PX if TEST_MODE else None, line_step=LINE_STEP,
               collection=COLLECTION.format(model=MODELS[0]))
    stored = {}
    if os.path.exists(path):
        with open(path) as f:
            doc = json.load(f)
        stored = doc.get("tiles", {}) if doc.get("made_with") == tag else {}
    new = [P for P in tiles if P["key"] not in stored]
    if new:
        with ThreadPoolExecutor(8) as ex:
            for P, (yes, month) in zip(new, ex.map(_has_data, new)):
                stored[P["key"]] = dict(has_data=bool(yes), month_read=month)
        local = os.path.join(NC_DIR, COVERAGE_NAME)
        with open(local, "w") as f:
            json.dump(dict(made_with=tag, created=iso(utcnow()), tiles=stored), f, indent=1)
        robust_drive_copy(local, path)
        os.remove(local)
    empty = sorted(P["key"] for P in tiles if not stored[P["key"]]["has_data"])
    if empty:
        log.info("%d tile(s) left out, OpenET holds no data for them (their ground is in the neighbouring zone's "
                 "tiles): %s", len(empty), " ".join(empty))
    return [P for P in tiles if stored[P["key"]]["has_data"]]


def build_tiles(recs):
    """Tiles with aquifer pixels: their frame and, by (year, month), every image of their UTM zone
    that covers them. Also notes which months OpenET has."""
    by_zone, grids = {}, {}
    for r in recs:
        by_zone.setdefault(r["crs"], []).append(r)
    cand, touching, odd = {}, {}, []
    for crs, images in sorted(by_zone.items()):
        ref = images[0]
        ox, oy = ref["tx"] % PIXEL, ref["ty"] % PIXEL
        for r in images:
            if not on_lattice(r, ref):
                odd.append(dict(image=r["id"], stage="not on its zone's 30 m grid"))
                continue
            g = (crs, r["tx"], r["ty"], r["w"], r["h"])
            if g not in grids:                   # the tiles an image grid reaches (a few grids serve every month)
                i0, i1 = math.floor((r["tx"] - ox) / TILE_M), math.ceil((r["tx"] + r["w"] * PIXEL - ox) / TILE_M) - 1
                j0, j1 = math.floor((r["ty"] - r["h"] * PIXEL - oy) / TILE_M), math.ceil((r["ty"] - oy) / TILE_M) - 1
                poly, keys = roi_polygon(crs), []
                for i in range(i0, i1 + 1):
                    for j in range(j0, j1 + 1):
                        left, top = ox + i * TILE_M, oy + (j + 1) * TILE_M
                        if poly.intersects(shapely.geometry.box(left, top - TILE_M, left + TILE_M, top)):
                            keys.append(tile_key(crs, i, j))
                            cand[keys[-1]] = (crs, ox, oy, i, j)
                grids[g] = keys
            for key in grids[g]:
                touching.setdefault(key, []).append(r)
    log_events(LOG_SKIPPED, odd)
    # A month counts as available when every chosen collection has it (months before a collection's
    # first month excepted), so that a month is not written while one of the models is still missing.
    by_model = {m: {(int(r["t"][:4]), int(r["t"][5:7])) for r in recs if r["model"] == m} for m in MODELS}
    complete = {k for s in by_model.values() for k in s
                if all(k in s2 or (s2 and k < min(s2)) for s2 in by_model.values())}
    AVAILABLE.clear()
    out = []
    for key, F in get_frames(cand).items():
        mask, months, hit = frame_mask(key, F), {}, {}
        for r in sorted(touching[key], key=lambda r: (r["t"], r["id"])):
            g = (r["tx"], r["ty"], r["w"], r["h"])
            if g not in hit:                     # does this image grid hold an aquifer pixel of the tile?
                S = scene_window(r, F)
                hit[g] = bool(S) and bool(mask[S["fr0"]:S["fr0"] + S["h"], S["fc0"]:S["fc0"] + S["w"]].any())
            if hit[g]:
                k = (int(r["t"][:4]), int(r["t"][5:7]))
                if k in complete:
                    months.setdefault(k, []).append(r)
                    AVAILABLE.add(k)
        if months:
            out.append(dict(key=key, F=F, months=months))
    out = with_data(out)
    out.sort(key=lambda P: (-P["F"]["n_roi"], P["key"]))
    return out


# ---------- where a scene lies on the frame, and its download blocks ----------
def scene_window(r, F):
    """The part of scene r that falls on frame F: dict(fr0, fc0 = its corner on the frame, h, w,
    sr0, sc0 = the same corner in the scene's own pixel grid). {} when they do not overlap, None when
    the scene is not on the frame's lattice (it is then left out, never resampled)."""
    dc, dr = (r["tx"] - F["x0"]) / PIXEL, (F["y0"] - r["ty"]) / PIXEL
    if r["crs"] != F["crs"] or r["sx"] != PIXEL or r["sy"] != -PIXEL or dc != int(dc) or dr != int(dr):
        return None
    dc, dr = int(dc), int(dr)
    fr0, fr1, fc0, fc1 = max(0, dr), min(F["h"], dr + r["h"]), max(0, dc), min(F["w"], dc + r["w"])
    if fr0 >= fr1 or fc0 >= fc1:
        return {}
    return dict(fr0=fr0, fc0=fc0, h=fr1 - fr0, w=fc1 - fc0, sr0=fr0 - dr, sc0=fc0 - dc)


# Earth Engine refuses a pixel request whose size, counted as pixels x bands x 3 bytes, is above 48 MiB
# (measured on Landsat's 16-bit bands: 1448 x 1448 pixels of 8 bands are served, 1449 x 1448 are refused).
# Should the count differ for these bands, a refused request is fetched in halves instead of failing.
EE_REQUEST_BYTES   = 48 * 1024 ** 2
EE_BYTES_PER_VALUE = 3


def max_block_pixels(n_bands):
    """Most pixels one download block of n_bands bands may hold: BLOCK_PX, cut back to Earth Engine's limit."""
    return max(1, min(int(BLOCK_PX) ** 2, EE_REQUEST_BYTES // (n_bands * EE_BYTES_PER_VALUE)))


def tile_scene(h, w, max_px):
    """Cut an h x w pixel rectangle into few equal rectangles of at most max_px pixels each; returns
    (row edges, column edges). Rectangles rather than squares: a 7991 x 7251 scene takes 28
    requests this way and 36 as squares, and the time goes into the number of requests. Long thin
    strips are avoided (side ratio at most 2) when that costs no more than a couple of requests,
    because compact blocks are more often skipped whole where the scene leaves the aquifer."""
    cands = []
    for nr in range(1, h + 1):
        bh = -(-h // nr)                         # tallest block with nr rows
        if bh <= max_px:
            nc = -(-w // (max_px // bh))         # fewest columns that keep bh x width within max_px
            bw = -(-w // nc)
            cands.append((nr * nc, max(bh, bw) / min(bh, bw), nr, nc))
            if nc == 1 or nr > 2 * min(c[0] for c in cands):
                break
    fewest = min(c[0] for c in cands)
    compact = [c for c in cands if c[1] <= 2.0 and c[0] <= fewest + max(2, fewest // 8)]
    _, _, nr, nc = min(compact or [c for c in cands if c[0] == fewest])
    return [i * h // nr for i in range(nr + 1)], [j * w // nc for j in range(nc + 1)]


def plan_blocks(S, mask, n_bands):
    """Download blocks of an image window: rectangles (row0, col0, h, w), in the window's own pixel
    count, that hold at least one aquifer pixel."""
    rows, cols = tile_scene(S["h"], S["w"], max_block_pixels(n_bands))
    return [(a, c, b - a, d - c) for a, b in zip(rows, rows[1:]) for c, d in zip(cols, cols[1:])
            if mask[S["fr0"] + a:S["fr0"] + b, S["fc0"] + c:S["fc0"] + d].any()]

In [ ]:
# ═══════════════ CELL 6: download the pixels of one month to scratch, check every block twice ═══════════════


def _grid(r, row0, col0, h, w):
    """A pixel rectangle of scene r on the scene's own grid: nothing is resampled on the way."""
    return dict(dimensions=dict(width=int(w), height=int(h)),
                affineTransform=dict(scaleX=r["sx"], shearX=0, translateX=r["tx"] + col0 * r["sx"],
                                     shearY=0, scaleY=r["sy"], translateY=r["ty"] + row0 * r["sy"]),
                crsCode=r["crs"])


def _check_block(arr, bands, shape):
    """A downloaded block must be exactly what was asked for: these bands, this shape, 16-bit
    integers (signed in release 2.0, unsigned in 2.1; the count bands of the models: unsigned 8-bit)."""
    if arr.dtype.names != tuple(bands):
        raise ValueError(f"bands {arr.dtype.names} instead of {tuple(bands)}")
    if any(arr.dtype[b] not in ((np.dtype("|u1"),) if b == "count" else (np.dtype("<i2"), np.dtype("<u2"))) for b in bands):
        raise ValueError(f"unexpected data type {arr.dtype}")
    if arr.shape != tuple(shape):
        raise ValueError(f"shape {arr.shape} instead of {tuple(shape)}")
    return arr


_SPLIT_NOTED = [False]


def fetch_array(r, bands, row0, col0, h, w, stop=None):
    """Earth Engine -> memory: one rectangle of scene r as the stored integers (getPixels, so no
    Earth Engine computation is involved). Returns (array, bytes received). A rectangle that Earth
    Engine refuses for its size is fetched as two halves and put together again, so a wrong guess
    about its limit costs a few extra requests and never the run."""
    def once():
        raw = ee.data.getPixels(dict(assetId=r["name"], bandIds=list(bands), grid=_grid(r, row0, col0, h, w),
                                     fileFormat="NPY"))
        return _check_block(np.load(io.BytesIO(raw), allow_pickle=False), bands, (h, w)), len(raw)   # refuse a damaged reply

    try:
        return with_retries(once, f"{r['id']} block {row0},{col0}", stop=stop)
    except RequestTooLarge as e:
        if h * w < 4096:
            raise IOError(f"{r['id']}: Earth Engine refuses even a {h} x {w} pixel request ({e})")
        if not _SPLIT_NOTED[0]:
            _SPLIT_NOTED[0] = True
            log.warning("Earth Engine refused a %d x %d pixel request of %d bands for its size (%s). Such blocks "
                        "are fetched in halves from now on; lowering BLOCK_PX avoids the extra requests.",
                        h, w, len(bands), e)
        if h >= w:
            k = h // 2
            top, n1 = fetch_array(r, bands, row0, col0, k, w, stop)
            bottom, n2 = fetch_array(r, bands, row0 + k, col0, h - k, w, stop)
            return np.concatenate([top, bottom], axis=0), n1 + n2
        k = w // 2
        left, n1 = fetch_array(r, bands, row0, col0, h, k, stop)
        right, n2 = fetch_array(r, bands, row0, col0 + k, h, w - k, stop)
        return np.concatenate([left, right], axis=1), n1 + n2


def download_block(r, S, blk, mm, stop):
    """One block of one image: fetch its bands in one request and put them in the image's scratch
    file (16 bits per value; the 8-bit count band is widened). Returns the block with a checksum of
    what was written."""
    row0, col0, h, w = blk
    arr, nbytes = fetch_array(r, r["bands"], S["sr0"] + row0, S["sc0"] + col0, h, w, stop)
    if stop.is_set():                        # the scratch folder may already belong to the next run
        raise RunStopping(r["id"])
    crc = 0
    for k, b in enumerate(r["bands"]):
        plane = np.ascontiguousarray(arr[b])
        if plane.dtype == np.dtype("<u2") and int(plane.max(initial=0)) > 32767:
            raise ValueError(f"{r['id']} band {b} holds a value above 32767, which the files cannot store")
        plane = plane.astype("<i2", copy=False)
        crc = zlib.crc32(plane, crc)
        mm[k, row0:row0 + h, col0:col0 + w] = plane
    return dict(blk=blk, crc=crc, bytes=nbytes)


def verify_scene(path, S, blocks, n_bands):
    """Read an image's scratch file back and compare every block with the checksum taken on arrival."""
    mm = np.load(path, mmap_mode="r")
    try:
        if mm.shape != (n_bands, S["h"], S["w"]) or mm.dtype != np.dtype("<i2"):
            raise IOError(f"{os.path.basename(path)}: unexpected layout {mm.dtype} {mm.shape}")
        for b in blocks:
            row0, col0, h, w = b["blk"]
            crc = 0
            for k in range(n_bands):
                crc = zlib.crc32(np.ascontiguousarray(mm[k, row0:row0 + h, col0:col0 + w]), crc)
            if crc != b["crc"]:
                raise IOError(f"{os.path.basename(path)}: block {row0},{col0} read back differently from what arrived")
    finally:
        del mm


def download_scenes(tag, scenes, F, mask, stop):
    """All blocks of the given images, N_DOWNLOAD_THREADS at a time, each image into one scratch
    file on its part of the frame. Returns (ok, permanent, transient): ok = [(image, part)], the
    others = [(image, reason)]. An image counts only when every block arrived, passed its check and
    was read back from scratch unchanged. Pixels that are not fetched (blocks without an aquifer
    pixel) are preset to OpenET's "no data" value, so they can never be taken for data."""
    plans = {}
    for r in scenes:
        S = scene_window(r, F)
        plans[r["id"]] = (S, plan_blocks(S, mask, len(r["bands"])))
    ensure_disk_free(f"{tag} download", extra_gb=sum(S["h"] * S["w"] * 2 * len(r["bands"]) for r in scenes for S in [plans[r["id"]][0]]) / 1024 ** 3)
    folder = os.path.join(RAW_DIR, tag)
    os.makedirs(folder, exist_ok=True)
    got, errors, futs, mms, paths = {r["id"]: [] for r in scenes}, {}, {}, {}, {}
    ex = ThreadPoolExecutor(N_DOWNLOAD_THREADS)
    try:
        with timed("download"):
            for r in scenes:
                S, blocks = plans[r["id"]]
                if not blocks:
                    continue
                paths[r["id"]] = os.path.join(folder, r["id"].replace("/", "_") + ".npy")
                mm = np.lib.format.open_memmap(paths[r["id"]], mode="w+", dtype="<i2", shape=(len(r["bands"]), S["h"], S["w"]))
                mm[:] = FILL                     # "no data" everywhere until a block says otherwise
                mms[r["id"]] = mm
                for blk in blocks:
                    futs[ex.submit(download_block, r, S, blk, mm, stop)] = r
            for fut in as_completed(futs):
                if stop.is_set():
                    raise RunStopping(tag)
                r = futs[fut]
                try:
                    got[r["id"]].append(fut.result())
                except SceneUnavailable as e:        # only Earth Engine saying "no such scene" is permanent
                    errors[r["id"]] = ("permanent", str(e))
                except Exception as e:
                    errors.setdefault(r["id"], ("transient", f"{type(e).__name__}: {e}"))
    except BaseException:
        stop.set()
        ex.shutdown(wait=False, cancel_futures=True)
        raise
    ex.shutdown(wait=True)
    for mm in mms.values():
        mm.flush()
    mm = fut = None                              # nothing may keep a scratch file open from here on: a failed
    mms.clear()                                  # request's traceback would, through the future that holds it
    futs.clear()
    gc.collect()
    ok, permanent, transient = [], [], []
    for r in scenes:
        S, blocks = plans[r["id"]]
        path = paths.get(r["id"])
        if r["id"] not in errors and blocks:
            try:
                with timed("download"):
                    verify_scene(path, S, got[r["id"]], len(r["bands"]))
            except Exception as e:
                errors[r["id"]] = ("transient", f"{type(e).__name__}: {e}")
        if r["id"] in errors:
            kind, why = errors[r["id"]]
            (permanent if kind == "permanent" else transient).append((r, why))
            if path and os.path.exists(path):
                os.remove(path)
        elif blocks:
            nbytes = sum(b["bytes"] for b in got[r["id"]])
            TOTALS["bytes"] = TOTALS.get("bytes", 0) + nbytes
            TOTALS["blocks"] = TOTALS.get("blocks", 0) + len(blocks)
            ok.append((r, dict(S, path=path, bytes=nbytes, blocks=len(blocks))))
        else:
            ok.append((r, None))                 # the scene touches the frame but none of its aquifer pixels
    return ok, permanent, transient


def fetch_images(tag, scenes, F, mask, stop):
    """Every image of one month that covers one tile, on scratch. An image that cannot be completed is
    tried again with growing pauses; after STALL_MINUTES without any progress the run stops
    cleanly. Returns (parts, permanent): parts = [(scene, part)] in time order."""
    todo = [r for r in scenes if r["id"] not in SKIP_IMAGES]
    permanent = [(r, "listed in SKIP_IMAGES") for r in scenes if r["id"] in SKIP_IMAGES]
    parts, waits, since = [], 0, None
    while todo:
        ok, perm, transient = download_scenes(tag, todo, F, mask, stop)
        parts += ok
        permanent += perm
        todo = [r for r, _ in transient]
        if ok or perm:
            waits, since = 0, None
        if todo:
            r0_, why0 = min(transient, key=lambda x: x[0]["t"])
            log.warning("[%s] %d image(s) could not be completed; first: %s (%s)", tag, len(todo), r0_["id"], why0[:200])
            since = since or time.time()
            if time.time() - since > STALL_MINUTES * 60:
                raise PipelineStop(
                    f"No progress for {STALL_MINUTES} min; stuck at {r0_['id']} ({why0[:200]}). Everything on Drive "
                    "is safe - re-run to resume. If this same image blocks every run while Earth Engine is "
                    "otherwise fine, add its id to SKIP_IMAGES.")
            waits += 1
            if stop.wait(min(30 * 2 ** waits, 600)):
                raise RunStopping(tag)
    return sorted(parts, key=lambda x: (x[0]["t"], x[0]["id"])), permanent


# ---------- which pixels hold data (release 2.1) ----------
# Release 2.0 stores -32768 where it has no data. Release 2.1 stores unsigned numbers without such a
# reserved value: a pixel without data and a pixel with 0 mm both arrive as 0 in a plain download. For
# those images the mask itself is read from Earth Engine, as one small computed image per block that
# holds one bit per collection. This is the only computed read of the notebook.
def _mask_image(scenes):
    """One image whose bit k is set where collection MODELS[k] has data (the mask of its first band)."""
    out = ee.Image.constant(0)
    for k, model in enumerate(MODELS):
        ims = [ee.Image(r["name"]).select(r["bands"][0]).mask().gt(0) for r in scenes if r["model"] == model]
        if ims:
            out = out.add(ee.ImageCollection.fromImages(ims).max().unmask(0).multiply(2 ** k))
    return out.toUint8().rename("bits")


def _compute_bits(img, crs, x0, y0, h, w, what, stop=None):
    """A rectangle of the mask image on the frame's own grid; fetched in halves if it is refused for its size."""
    grid = dict(dimensions=dict(width=int(w), height=int(h)), crsCode=crs,
                affineTransform=dict(scaleX=PIXEL, shearX=0, translateX=x0, shearY=0, scaleY=-PIXEL, translateY=y0))
    try:
        a = with_retries(lambda: ee.data.computePixels(dict(expression=img, grid=grid, fileFormat="NUMPY_NDARRAY")),
                         what, stop=stop)["bits"]
        if a.shape != (h, w):
            raise IOError(f"{what}: shape {a.shape} instead of {(h, w)}")
        return a.astype("u1")
    except RequestTooLarge:
        if h >= w:
            k = h // 2
            return np.concatenate([_compute_bits(img, crs, x0, y0, k, w, what, stop),
                                   _compute_bits(img, crs, x0, y0 - k * PIXEL, h - k, w, what, stop)], axis=0)
        k = w // 2
        return np.concatenate([_compute_bits(img, crs, x0, y0, h, k, what, stop),
                               _compute_bits(img, crs, x0 + k * PIXEL, y0, h, w - k, what, stop)], axis=1)


def fetch_bits(tag, scenes, F, mask, stop):
    """The data mask of every collection on the frame, as bits (see above); 0 where no block was read."""
    img = _mask_image(scenes)
    bits = np.zeros((F["h"], F["w"]), dtype="u1")
    rows, cols = tile_scene(F["h"], F["w"], int(MASK_BLOCK_PX) ** 2)
    blocks = [(a, c, b - a, d - c) for a, b in zip(rows, rows[1:]) for c, d in zip(cols, cols[1:]) if mask[a:b, c:d].any()]
    with timed("mask read (computed)"), ThreadPoolExecutor(min(8, max(1, len(blocks)))) as ex:
        futs = {ex.submit(_compute_bits, img, F["crs"], F["x0"] + c * PIXEL, F["y0"] - a * PIXEL, h, w, f"{tag} mask", stop):
                (a, c, h, w) for a, c, h, w in blocks}
        for fut in as_completed(futs):
            a, c, h, w = futs[fut]
            try:
                bits[a:a + h, c:c + w] = fut.result()
            except RunStopping:
                raise
            except Exception as e:
                raise PipelineStop(f"The data mask of {tag} could not be read ({type(e).__name__}: {str(e)[:200]}). "
                                   "Everything on Drive is safe - re-run to resume.")
    TOTALS["mask_blocks"] = TOTALS.get("mask_blocks", 0) + len(blocks)
    return bits


def fetch_month(tag, scenes, F, mask, stop):
    """Everything one month of one tile needs: the images (fetch_images) and, for images without a
    reserved "no data" number, the mask bits. Returns (parts, permanent) like fetch_images; each part
    of such an image carries the bits."""
    parts, permanent = fetch_images(tag, scenes, F, mask, stop)
    need = [r for r, part in parts if part is not None and not r["signed"]]
    if need:
        bits = fetch_bits(tag, need, F, mask, stop)
        for r, part in parts:
            if part is not None and not r["signed"]:
                part["bits"] = bits
    return parts, permanent


def valid_pixels(r, band, row0, col0, h, w):
    """True where image r has data in a rectangle of its own grid (signed images: the stored number
    says so; unsigned ones: Earth Engine's mask is read)."""
    if r["signed"]:
        return fetch_array(r, [band], row0, col0, h, w)[0][band] != FILL
    img = ee.Image(r["name"]).select(band).mask().gt(0).unmask(0).toUint8().rename("bits")
    return _compute_bits(img, r["crs"], r["tx"] + col0 * PIXEL, r["ty"] - row0 * PIXEL, h, w, f"mask of {r['id']}") > 0

In [ ]:
# ═══════════════ CELL 7: place the downloaded pixels on the tile; check against Earth Engine's own reading ═══════════════
def place_strip(parts, mask_strip, r0, r1, c0, c1):
    """Values of frame rows r0..r1, columns c0..c1 for every variable, exactly as OpenET stores them.

    parts = [(image, part)] of the month; part["mm"] holds the image's bands on its window of the
    frame. Images are laid on the frame by whole-pixel shifts; nothing is computed, scaled or
    resampled. Pixels outside the aquifer polygon, and pixels no image covers, hold the fill value."""
    h, w = r1 - r0, c1 - c0
    out = {v: np.full((h, w), VAR_FILL[v], dtype=VAR_TYPE[v]) for v in VARS}
    for r, part in parts:
        if part is None:
            continue
        a0, a1 = max(r0, part["fr0"]), min(r1, part["fr0"] + part["h"])      # overlap, in frame rows / columns
        b0, b1 = max(c0, part["fc0"]), min(c1, part["fc0"] + part["w"])
        if a0 >= a1 or b0 >= b1:
            continue
        blk = np.asarray(part["mm"][:, a0 - part["fr0"]:a1 - part["fr0"], b0 - part["fc0"]:b1 - part["fc0"]])
        win = (slice(a0 - r0, a1 - r0), slice(b0 - c0, b1 - c0))
        inside = mask_strip[win]
        if not r["signed"]:                      # release 2.1: only pixels Earth Engine's mask marks as data
            bit = MODELS.index(r["model"])
            inside = inside & ((part["bits"][a0:a1, b0:b1] >> bit) & 1).astype(bool)
        for k, band in enumerate(r["bands"]):
            v, plane = var_name(r["model"], band), blk[k]
            if IS_COUNT[v]:                      # widened to 16 bits on scratch; a block never fetched holds FILL
                plane = np.where(plane == FILL, NO_COUNT, plane).astype("u1")
            out[v][win] = np.where(inside, plane, out[v][win])
    return out


def cross_check(r):
    """A small window of image r read twice: by this notebook's download (stored integers on the
    image's own grid) and by Earth Engine's own evaluation of the image, with its mask. They must
    be identical where the image has data, and the download must hold "no data" where it has none.
    This proves that the download path neither resamples nor shifts the pixels."""
    n = CHECK_PX
    row0, col0 = (r["h"] - n) // 2, (r["w"] - n) // 2
    mine, _ = fetch_array(r, r["bands"], row0, col0, n, n)
    g = _grid(r, row0, col0, n, n)
    img = ee.Image(r["name"].split("/assets/", 1)[1] if r["name"].startswith("projects/earthengine-public") else r["name"])
    both = img.select(r["bands"]).unmask(-9999).toInt32()
    theirs = with_retries(lambda: ee.data.computePixels(dict(expression=both, grid=g, fileFormat="NUMPY_NDARRAY")),
                          "cross-check")
    first = theirs[r["bands"][0]] != -9999
    for b in r["bands"]:
        a, t = mine[b].astype("i4"), theirs[b]
        has = t != -9999
        if not np.array_equal(a[has], t[has]):
            raise DataIntegrityError(f"{r['id']} band {b}: the downloaded pixels differ from Earth Engine's own reading")
        if not r["signed"] and not np.array_equal(has, first):
            raise DataIntegrityError(f"{r['id']}: band {b} has another mask than band {r['bands'][0]}; the notebook "
                                     "takes one mask per image")
        if r["signed"] and b != "count" and np.count_nonzero(a[~has] != FILL):
            raise DataIntegrityError(f"{r['id']} band {b}: pixels without data did not arrive as {FILL}")
    if not r["signed"]:
        mine_mask = valid_pixels(r, r["bands"][0], row0, col0, n, n)
        if not np.array_equal(mine_mask, first):
            raise DataIntegrityError(f"{r['id']}: the mask read by the notebook differs from Earth Engine's own")
    log.info("Cross-check OK on %s: %d x %d pixels of %d bands identical to Earth Engine's own reading", r["id"], n, n,
             len(r["bands"]))
    return dict(image=r["id"], pixels=n * n)

In [ ]:
# ═══════════════ CELL 8: the yearly CF NetCDF file: writer, state, read-back check, audit ═══════════════
SETTINGS_SIGNATURE = json.dumps(dict(variables=VARS, collection=COLLECTION, pixel=PIXEL, fill=[FILL, NO_COUNT],
                                     content="monthly totals as stored"), sort_keys=True)
DRIVE_RETRY_S = 20                       # pause before a file on Drive is read again
STRIP = max(1, STRIP_ROWS // CHUNK_YX) * CHUNK_YX      # strips are whole rows of chunks, so each chunk is written once
TILE_RE = re.compile(r"z\d\de\d+n\d+")


def unit_dir(key, year):
    """Folder of one tile's file of one year: <output folder>/<year>/<tile>/"""
    return os.path.join(DRIVE_OUT, str(year), key)


def unit_name(key, year):
    return f"{PRODUCT_TAG}_{key}_{year}_{ROI_LABEL}.nc"


def _cf_attr(v):
    if isinstance(v, (str, int, float, np.number, np.ndarray)):
        return v
    if isinstance(v, (list, tuple)):
        return np.asarray(v)
    return str(v)


def create_year_nc(path, P, year, mask):
    """Create the empty CF file of one tile and year on the tile's fixed 30 m frame.
    Written under a temporary name and renamed when complete."""
    F, key = P["F"], P["key"]
    crs = pyproj.CRS.from_user_input(F["crs"])
    zkw = dict(zlib=COMPRESS_LEVEL > 0, complevel=max(COMPRESS_LEVEL, 1), shuffle=True)
    cy, cx = min(CHUNK_YX, F["h"]), min(CHUNK_YX, F["w"])
    tmp = path + ".new"
    ds = nc4.Dataset(tmp, "w", format="NETCDF4")
    try:
        ds.createDimension("time", None)
        ds.createDimension("y", F["h"])
        ds.createDimension("x", F["w"])
        ds.createDimension("nv", 2)
        tv = ds.createVariable("time", "f8", ("time",), chunksizes=(12,))
        tv.units, tv.calendar, tv.standard_name, tv.axis = TIME_UNITS, TIME_CALENDAR, "time", "T"
        tv.long_name, tv.bounds = "first day of the month (UTC)", "time_bnds"
        ds.createVariable("time_bnds", "f8", ("time", "nv"), chunksizes=(12, 2))
        yv = ds.createVariable("y", "f8", ("y",))
        yv.standard_name, yv.units, yv.axis = "projection_y_coordinate", "m", "Y"
        yv.long_name = "northing of pixel centre"
        yv[:] = F["y0"] - (np.arange(F["h"]) + 0.5) * PIXEL
        xv = ds.createVariable("x", "f8", ("x",))
        xv.standard_name, xv.units, xv.axis = "projection_x_coordinate", "m", "X"
        xv.long_name = "easting of pixel centre"
        xv[:] = F["x0"] + (np.arange(F["w"]) + 0.5) * PIXEL
        gm = ds.createVariable("crs", "i4")
        for k, v in crs.to_cf().items():
            gm.setncattr(k, _cf_attr(v))
        gm.spatial_ref = crs.to_wkt()
        gm.GeoTransform = f"{F['x0']} {PIXEL} 0.0 {F['y0']} 0.0 {-PIXEL}"       # GDAL order
        gm.long_name = f"OpenET's own 30 m grid of UTM zone {int(key[1:3])} ({F['crs']}), tile {key}"

        mv = ds.createVariable("roi_mask", "i1", ("y", "x"), chunksizes=(cy, cx), **zkw)
        mv.long_name = f"1 where the pixel centre lies inside the {ROI_LABEL} aquifer polygon"
        mv.flag_values = np.array([0, 1], dtype="i1")
        mv.flag_meanings = "outside_roi inside_roi"
        mv.grid_mapping = "crs"
        for r0 in range(0, F["h"], 2048):
            mv[r0:r0 + 2048, :] = mask[r0:r0 + 2048].astype("i1")

        for m in MODELS:
            for band in BANDS[m]:
                name = var_name(m, band)
                v = ds.createVariable(name, VAR_TYPE[name], ("time", "y", "x"), chunksizes=(1, cy, cx),
                                      fill_value=(np.uint8(NO_COUNT) if IS_COUNT[name] else np.int16(FILL)), **zkw)
                v.grid_mapping = "crs"
                v.source_collection, v.source_band = COLLECTION.format(model=m), band
                if m == "ensemble":
                    v.long_name = ENSEMBLE_BANDS[band]
                    v.units = "1" if band.endswith(("_count", "_index")) else "mm"
                elif band == "et":
                    v.long_name = f"evapotranspiration of the month, {MODEL_NAME[m]} model"
                    v.units = "mm"
                else:
                    v.long_name = f"number of cloud-free Landsat observations the {MODEL_NAME[m]} value of the month rests on"
                    v.units = "1"
                    v.comment = (f"As stored by OpenET. {NO_COUNT} marks pixels outside the polygon, pixels no OpenET "
                                 "image covers and, from release 2.1 on, pixels where OpenET has no data (release 2.0 "
                                 "itself stores 0 there).")
                if v.units == "mm":
                    v.cell_methods = "time: sum"
                    v.comment = ("Monthly total in whole millimetres, exactly the integer OpenET stores (its scale "
                                 "factor is 1). _FillValue: OpenET has no data (its own mask), the pixel is outside the "
                                 "polygon, or no OpenET image covers it. 0 is a real value of 0 mm.")

        ds.createVariable("n_images", "i2", ("time",), chunksizes=(12,)).long_name = "OpenET images read for the month"
        ds.createVariable("source_images", str, ("time",)).long_name = "ids of the OpenET images read for the month"

        ds.Conventions = "CF-1.9"
        ds.title = (f"OpenET {OPENET_VERSION} monthly evapotranspiration, {ROI_LABEL} aquifer, UTM zone "
                    f"{int(key[1:3])} tile {key}, {year}: ensemble and six models at 30 m")
        ds.summary = ("One record per month. Every variable is a band of an OpenET monthly image, copied as the "
                      "stored integers onto a tile of OpenET's own 30 m grid by whole-pixel shifts: nothing is "
                      "computed, scaled, resampled or gap-filled here. The tiles are those of the Landsat index "
                      "files of this project, so the two line up pixel for pixel.")
        ds.institution = "OpenET (source data); packaged by the MSU groundwater project"
        ds.source = (f"OpenET {OPENET_VERSION} monthly images read from Google Earth Engine: " +
                     ", ".join(COLLECTION.format(model=m) for m in MODELS))
        ds.references = ("Melton et al. 2022, https://doi.org/10.1111/1752-1688.12956; Volk et al. 2024, "
                         "https://doi.org/10.1038/s44221-023-00181-7; https://etdata.org")
        ds.license = "CC-BY-4.0 (OpenET)"
        ds.history = f"{iso(utcnow())} created by OpenET_Monthly_30m_Ogallala.ipynb"
        ds.openet_version = OPENET_VERSION
        ds.tile, ds.utm_zone, ds.year, ds.roi = key, np.int32(int(key[1:3])), np.int32(year), ROI_LABEL
        ds.variables_list = " ".join(VARS)
        ds.frame = json.dumps(dict(crs=F["crs"], x0=F["x0"], y0=F["y0"], w=F["w"], h=F["h"]))
        ds.settings_signature = SETTINGS_SIGNATURE
        ds.processing_status = "in_progress"
        ds.months = ""
        ds.committed_records = np.int32(0)
    finally:
        ds.close()
    os.replace(tmp, path)


def read_state(path, tries=1):
    """(state, None) of a yearly file, or (None, why) when it cannot be used: unreadable, a record
    count that disagrees with its last commit, or months out of order. From Drive it is tried
    several times: a Drive that is slow or only just mounted must never be mistaken for a damaged file."""
    why = None
    for attempt in range(max(1, tries)):
        if attempt:
            time.sleep(DRIVE_RETRY_S * attempt)
        try:
            with nc4.Dataset(path, "r") as ds:
                n = len(ds.variables["time"])
                months = [int(m) for m in str(getattr(ds, "months", "")).split()]
                if int(getattr(ds, "committed_records", -1)) != n or len(months) != n:
                    why = (f"holds {n} records but its last commit says {getattr(ds, 'committed_records', '?')} "
                           "(an interrupted write)")
                elif months != sorted(set(months)):
                    why = "has its months out of order"
                else:
                    return dict(n=n, months=months, status=str(getattr(ds, "processing_status", "in_progress")),
                                signature=str(getattr(ds, "settings_signature", "")),
                                frame=json.loads(str(getattr(ds, "frame", "{}"))), size=os.path.getsize(path)), None
        except Exception as e:
            why = f"cannot be read ({type(e).__name__}: {str(e)[:160]})"
        log.warning("%s %s (attempt %d of %d)", os.path.basename(path), why, attempt + 1, max(1, tries))
    return None, why


class YearWriter:
    """Single writer of one yearly file on scratch. A month is written strip by strip and counted
    only when finish_month() has stored it in the file's own list of months."""

    def __init__(self, path):
        self.path, self.ds = path, None
        self.ds = nc4.Dataset(self.path, "a")
        self.ds.set_auto_maskandscale(False)
        self.n = len(self.ds.variables["time"])
        self.months = [int(m) for m in str(self.ds.months).split()]

    def write_strip(self, r0, r1, c0, c1, arrays):
        for name, a in arrays.items():
            if np.count_nonzero(a != VAR_FILL[name]):        # an array of fill values stays unwritten
                self.ds.variables[name][self.n, r0:r1, c0:c1] = a

    def _in_order(self, month):
        if self.months and month <= self.months[-1]:
            raise DataIntegrityError(f"{os.path.basename(self.path)}: month {month} after month {self.months[-1]}")

    def finish_month(self, year, month, images, status=None):
        self._in_order(month)
        v, i = self.ds.variables, self.n
        start, end = month_bounds(year, month)
        v["time"][i] = (start - EPOCH).days
        v["time_bnds"][i, :] = [(start - EPOCH).days, (end - EPOCH).days]
        v["n_images"][i] = len(images)
        v["source_images"][i] = " ".join(r["id"] for r in images)
        self.n, self.months = i + 1, self.months + [month]
        self.commit(status)

    def take_month(self, old, t_old, month):
        """Count a month whose arrays were copied from another file of this tile and year; its
        record (time, images read) is copied from there as well."""
        self._in_order(month)
        v, o, i = self.ds.variables, old.variables, self.n
        for name in ("time", "time_bnds", "n_images", "source_images"):
            v[name][i] = o[name][t_old]
        self.n, self.months = i + 1, self.months + [month]
        self.commit()

    def commit(self, status=None):
        ds = self.ds
        ds.months = " ".join(str(m) for m in self.months)
        ds.committed_records = np.int32(self.n)
        if status:
            ds.processing_status = status
        if self.n:
            ds.time_coverage_start = iso(month_bounds(int(ds.year), self.months[0])[0])
            ds.time_coverage_end = iso(month_bounds(int(ds.year), self.months[-1])[1])
        ds.sync()

    def close(self):
        if self.ds is not None:
            self.ds.close()
            self.ds = None


def strip_windows(mask):
    """The parts of a frame that are written, as (r0, r1, c0, c1): strips of rows, their columns cut
    to the chunks that hold aquifer pixels. Rows without any aquifer pixel are left out, so that
    their chunks stay unwritten."""
    ny, nx = mask.shape
    for r0 in range(0, ny, STRIP):
        r1 = min(ny, r0 + STRIP)
        cols = np.flatnonzero(mask[r0:r1].any(axis=0))
        if cols.size:
            yield r0, r1, int(cols[0]) // CHUNK_YX * CHUNK_YX, min(nx, -(-(int(cols[-1]) + 1) // CHUNK_YX) * CHUNK_YX)


def copy_month(old, t_old, month, writer, mask):
    """One month of another file of the same tile and year into the file being written, exactly as
    it is stored there. Returns the checksums of what was read, so that the closed file can be
    compared with it."""
    crcs = {}
    for r0, r1, c0, c1 in strip_windows(mask):
        arrays = {name: np.ascontiguousarray(old.variables[name][t_old, r0:r1, c0:c1]) for name in VARS}
        writer.write_strip(r0, r1, c0, c1, arrays)
        for name, a in arrays.items():
            crcs[(name, r0, r1, c0, c1)] = zlib.crc32(a)
    writer.take_month(old, t_old, month)
    return crcs


def readback_month(path, t, crcs):
    """Read one month back from the closed file, strip by strip, and compare each array's checksum
    with the one taken when it was computed."""
    with nc4.Dataset(path, "r") as ds:
        ds.set_auto_maskandscale(False)
        for (name, r0, r1, c0, c1), want in crcs.items():
            got = np.ascontiguousarray(ds.variables[name][t, r0:r1, c0:c1])
            if zlib.crc32(got) != want:
                raise DataIntegrityError(f"{os.path.basename(path)} record {t} {name} rows {r0}-{r1}: the file "
                                         "differs from what was computed")
    return len(crcs)


def audit_nc(path, mask=None):
    """Read-only audit of a yearly file. Returns (ok, problems, info). With the mask, three strips
    of the first and last month are also checked against it."""
    problems, info = [], {}
    try:
        with nc4.Dataset(path, "r") as ds:
            ds.set_auto_maskandscale(False)
            if not str(getattr(ds, "Conventions", "")).startswith("CF-"):
                problems.append("no CF Conventions attribute")
            names = str(getattr(ds, "variables_list", "")).split()
            need = ["time", "time_bnds", "y", "x", "crs", "roi_mask", "n_images", "source_images"] + names
            missing = [v for v in need if v not in ds.variables]
            if missing or not names:
                return False, [f"missing variables {missing}"], info
            n = len(ds.variables["time"])
            months = [int(m) for m in str(getattr(ds, "months", "")).split()]
            t = np.asarray(ds.variables["time"][:], dtype="f8")
            if int(getattr(ds, "committed_records", -1)) != n or len(months) != n:
                problems.append("record count differs from the committed months")
            if months != sorted(set(months)) or (n > 1 and not np.all(np.diff(t) > 0)):
                problems.append("months not strictly increasing")
            year = int(getattr(ds, "year", 0))
            if n and [(month_bounds(year, m)[0] - EPOCH).days for m in months] != [int(v) for v in t]:
                problems.append("time values do not match the months")
            for name in names:
                v = ds.variables[name]
                if v.dimensions != ("time", "y", "x") or getattr(v, "grid_mapping", None) != "crs":
                    problems.append(f"{name}: unexpected dimensions or no grid_mapping")
            ny, nx = len(ds.dimensions["y"]), len(ds.dimensions["x"])
            if mask is not None and n:
                if mask.shape != (ny, nx):
                    problems.append("the file is not on the tile's frame")
                else:
                    for i in sorted({0, n - 1}):
                        for r0 in sorted({0, (ny // 2 // STRIP) * STRIP, ((ny - 1) // STRIP) * STRIP}):
                            m = mask[r0:r0 + STRIP]
                            for name in (names[0], names[-1]):
                                a = np.asarray(ds.variables[name][i, r0:r0 + STRIP, :])
                                if np.count_nonzero(a[~m] != ds.variables[name]._FillValue):
                                    problems.append(f"record {i} rows {r0}+: {name} has values outside the polygon")
            info.update(records=n, months=months, status=str(getattr(ds, "processing_status", "?")),
                        shape=f"{ny}x{nx}", size=os.path.getsize(path))
    except Exception as e:
        problems.append(f"cannot be opened/read: {type(e).__name__}: {e}")
    return (not problems), problems, info

In [ ]:
# ═══════════════ CELL 9: year by year, tile by tile: one yearly file per unit; run ═══════════════
# How a run keeps its place. The unit of work is one tile and one year. Its file is built on scratch, checked,
# and only then copied to Drive under its final name, in one verified step. Nothing on Drive is ever half
# written, appended to or compressed twice. A runtime that dies loses the unit it was working on and nothing
# else; the next run starts with the first file that is missing or lacks a month of the range.
#
# Which months a run makes is set by START_MONTH and STOP_MONTH. A yearly file holds the months made so far.
# When a later run wants more months of that year, the file on Drive is fetched and the new months are
# added to it; a month that belongs before one already there is written into a new file, into which the
# months on Drive are copied as they are stored. Either way a month that is on Drive is never fetched again.
SIG_ID = hashlib.md5(SETTINGS_SIGNATURE.encode()).hexdigest()[:12]
LAST = {}                       # the last file that reached Drive (for the message printed at a stop)
ALL_TILES = []


def frame_id(F):
    return [F["crs"], F["x0"], F["y0"], F["w"], F["h"]]


def load_ledger():
    """Yearly files already on Drive, as noted when each was uploaded (the last note of a file counts)."""
    led, path = {}, os.path.join(DRIVE_OUT, LEDGER_NAME)
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                try:
                    e = json.loads(line)
                    led[e["file"]] = e
                except (ValueError, KeyError):
                    pass
    return led


def save_to_drive(src, dst, what):
    """Verified copy to Drive; a Drive that will not take the file stops the run cleanly."""
    try:
        robust_drive_copy(src, dst)
    except PipelineStop:
        raise
    except Exception as e:
        raise PipelineStop(f"{what} could not be saved to Drive ({e}). Everything saved before it is still "
                           "there - re-run to resume.")


class Progress:
    def __init__(self, total):
        self.total, self.done, self.run, self.t0 = total, 0, 0, time.time()

    def add(self, n, resumed=False):
        self.done += n
        if not resumed:
            self.run += n

    def line(self):
        rate = self.run / max(time.time() - self.t0, 1e-9)
        eta = (self.total - self.done) / rate / 3600 if rate > 0 else float("nan")
        return (f"{self.done}/{self.total} image pieces ({100 * self.done / max(1, self.total):.1f} %), "
                f"{rate * 60:.1f}/min, ETA {eta:.1f} h")


def unit_plan(P, year, ledger):
    """What one tile-year needs: nothing (its file on Drive holds every month this run wants), a
    new file, or more months for the file that is there ("append" when they all come after its
    last month, "merge" when one belongs before it). A file that cannot be read stops the run: it
    is never replaced or redone because of that."""
    key, name = P["key"], unit_name(P["key"], year)
    drive_nc = os.path.join(unit_dir(key, year), name)
    want = wanted_months(year)
    if not want:
        return dict(action="skip", months=[])
    if not os.path.exists(drive_nc):
        if name in ledger:
            log.warning("%s was uploaded earlier but is not on Drive now - it is made again", name)
        return dict(action="new", todo=want, have=[], drive_nc=drive_nc)
    led = ledger.get(name)
    if led and led.get("size") == os.path.getsize(drive_nc) and led.get("settings") == SIG_ID \
            and led.get("frame") == frame_id(P["F"]):
        have = [int(m) for m in led["months"]]
    else:
        st, why = read_state(drive_nc, DRIVE_READ_TRIES)
        if st is None:
            raise PipelineStop(
                f"{name} is on Drive but {why}. Nothing was changed or deleted. Run again in a few minutes (Drive "
                "can be slow just after it is mounted). If the file really is damaged, move it out of the folder "
                "yourself and run again: only that tile-year is then made again.")
        if st["signature"] != SETTINGS_SIGNATURE:
            raise PipelineStop(f"{name} was made with other settings (the list of collections). "
                               "Restore those settings in Cell 2, or use an empty output folder.")
        if [st["frame"].get(k) for k in ("crs", "x0", "y0", "w", "h")] != frame_id(P["F"]):
            raise PipelineStop(f"{name} is not on the frame stored in {FRAMES_NAME}. Nothing was changed.")
        have = st["months"]
        ledger[name] = dict(file=name, tile=key, year=year, months=have, size=st["size"], settings=SIG_ID,
                            frame=frame_id(P["F"]), note="found on Drive")
        log_events(LEDGER_NAME, [ledger[name]])
    todo = [m for m in want if m not in have]
    if not todo:
        return dict(action="skip", months=have)
    action = "append" if not have or min(todo) > max(have) else "merge"
    return dict(action=action, todo=todo, have=have, drive_nc=drive_nc)


def compose_month(parts, mask, writer, keep_checksums):
    """One month onto the yearly file, strip by strip. Returns (checksums or None, counters)."""
    opened = [(r, None if part is None else dict(part, mm=np.load(part["path"], mmap_mode="r"))) for r, part in parts]
    crcs = {} if keep_checksums else None
    px = dict(covered=0)
    try:
        for r0, r1, c0, c1 in strip_windows(mask):
            arrays = place_strip(opened, mask[r0:r1, c0:c1], r0, r1, c0, c1)
            writer.write_strip(r0, r1, c0, c1, arrays)
            px["covered"] += int(np.count_nonzero(arrays[MAIN_VAR] != VAR_FILL[MAIN_VAR]))
            if crcs is not None:
                for name, a in arrays.items():
                    crcs[(name, r0, r1, c0, c1)] = zlib.crc32(np.ascontiguousarray(a))
    finally:
        del opened
    return crcs, px


def process_unit(P, year, plan, prog):
    """One tile and year: fetch a month (the next one is fetched while this one is computed and
    written), compose it, repeat; then check the closed file and copy it to Drive. Months that are
    on Drive already are kept as they are: the file is added to ("append") or, when a new month
    belongs before one of them, they are copied as stored into a new file ("merge")."""
    key, F = P["key"], P["F"]
    name, drive_nc = unit_name(key, year), plan["drive_nc"]
    tag = f"{key}_{year}"
    work, raw = os.path.join(NC_DIR, tag), os.path.join(RAW_DIR, tag)
    for d in (work, raw):
        shutil.rmtree(d, ignore_errors=True)     # scratch left by an earlier run is never used
    os.makedirs(work, exist_ok=True)
    local, t_start, months = os.path.join(work, name), time.time(), plan["todo"]
    kept = plan["have"] if plan["action"] == "merge" else []      # months copied from the file on Drive
    mask = frame_mask(key, F)
    scenes_of = lambda m: P["months"].get((year, m), [])
    stop, pool, writer, old = threading.Event(), ThreadPoolExecutor(1), None, None
    fetch = lambda m: fetch_month(f"{tag}/m{m:02d}", scenes_of(m), F, mask, stop)
    check = set(months) if TEST_MODE else set(months[:READBACK_MONTHS])
    counts, checks = dict(scenes=0, failed=0), {}
    try:
        try:
            on_drive = os.path.getsize(drive_nc) if plan["action"] != "new" else 0
            ensure_disk_free(f"{tag} file", extra_gb=(F["n_roi"] * 12 * (len(months) + len(kept)) + on_drive) / 1024 ** 3)
            header = None
            if plan["action"] != "new":
                fetched = local if plan["action"] == "append" else os.path.join(work, "on_drive.nc")
                restore_from_drive(drive_nc, fetched)
                st, why = read_state(fetched)
                if st is None or st["months"] != plan["have"]:
                    raise PipelineStop(f"The copy of {name} fetched from Drive is not the file that was read a "
                                       f"moment ago ({why or 'it has changed'}). Nothing was changed - run again.")
                if st["signature"] != SETTINGS_SIGNATURE:
                    raise PipelineStop(f"{name} was made with other settings (the list of collections). Restore "
                                       "those settings in Cell 2. Nothing was changed.")
            if plan["action"] != "append":
                create_year_nc(local, P, year, mask)
            if plan["action"] == "new":
                header = os.path.getsize(local)
            writer = YearWriter(local)
            if kept:
                old = nc4.Dataset(fetched, "r")
                old.set_auto_maskandscale(False)
                if not np.array_equal(np.asarray(old.variables["roi_mask"][:]).astype(bool), mask):
                    raise PipelineStop(f"{name} on Drive was made with another aquifer mask than this run has. "
                                       "Nothing was changed.")
            fut, i = pool.submit(fetch, months[0]), 0
            for m in sorted(months + kept):
                if m in kept:                    # on Drive already: copied as stored, and compared after closing
                    t_c, t = time.time(), writer.n
                    checks[t] = copy_month(old, plan["have"].index(m), m, writer, mask)
                    TIMES["months copied"] = TIMES.get("months copied", 0.0) + time.time() - t_c
                    log.info("[%s-%02d] copied as stored from the file on Drive", tag, m)
                    continue
                parts, permanent = fut.result()
                i += 1
                fut = pool.submit(fetch, months[i]) if i < len(months) else None
                if permanent:
                    counts["failed"] += len(permanent)
                    log_events(LOG_FAILED, [dict(image=r["id"], tile=key, reason=why) for r, why in permanent])
                    for r, why in permanent:
                        log.warning("[%s] permanently unavailable: %s (%s)", tag, r["id"], why[:160])
                t_c, t = time.time(), writer.n
                crcs, px = compose_month(parts, mask, writer, m in check)
                used = [r for r, part in parts if part is not None]
                writer.finish_month(year, m, used)
                TIMES["place + write"] = TIMES.get("place + write", 0.0) + time.time() - t_c
                if crcs:
                    checks[t] = crcs
                shutil.rmtree(os.path.join(raw, f"m{m:02d}"), ignore_errors=True)
                counts["scenes"] += len(used)
                prog.add(len(scenes_of(m)))
                log.info("[%s-%02d] %d images, %.0f %% of the aquifer pixels have an ensemble value | %s | %s downloaded | free %.0f GB",
                         tag, m, len(used), 100 * px["covered"] / max(1, F["n_roi"]), prog.line(),
                         human(TOTALS.get("bytes", 0)), free_gb())
                gc.collect()
            status = "complete" if writer.months == list(range(1, 13)) else ("test" if TEST_MODE else "partial")
            writer.commit(status)
            writer.close()
            if old is not None:
                old.close()
                old = None
        except BaseException:
            stop.set()                           # the month being fetched in the background stops too
            pool.shutdown(wait=False, cancel_futures=True)
            raise
        pool.shutdown(wait=True)
        # ---------- the closed file is checked before it may reach Drive ----------
        t_v = time.time()
        compared = sum(readback_month(local, t, crcs) for t, crcs in checks.items())
        good, problems, info = audit_nc(local, mask)
        if not good:
            raise DataIntegrityError(f"{name} failed its audit and was not uploaded: {problems}")
        TIMES["checks"] = TIMES.get("checks", 0.0) + time.time() - t_v
        save_to_drive(local, drive_nc, name)
        size = os.path.getsize(local)
        entry = dict(file=name, tile=key, year=year, months=info["months"], size=size, settings=SIG_ID,
                     frame=frame_id(F), status=info["status"], scenes=counts["scenes"], failed=counts["failed"],
                     header_bytes=header, seconds=round(time.time() - t_start))
        log_events(LEDGER_NAME, [entry])
        LAST.update(file=name)
        TOTALS["files"] = TOTALS.get("files", 0) + 1
        TOTALS["file_bytes"] = TOTALS.get("file_bytes", 0) + size
        log.info("[%s] DONE: months %s, %d images read, %s on Drive (%d arrays read back and compared), %.1f min", tag,
                 span(info["months"]), counts["scenes"], human(size), compared, (time.time() - t_start) / 60)
        return dict(entry, action=plan["action"], n_roi=F["n_roi"], new_months=len(months))
    finally:
        for f in (writer, old):
            try:
                if f is not None:
                    f.close()
            except Exception:
                pass
        shutil.rmtree(raw, ignore_errors=True)
        shutil.rmtree(work, ignore_errors=True)


def preflight(units):
    """Fail fast before any bulk work: Drive is writable, the disk floor holds, and for every
    collection the download returns exactly the pixels Earth Engine itself reads."""
    local, remote = os.path.join(NC_DIR, "_write_test.txt"), os.path.join(DRIVE_OUT, "_write_test.txt")
    with open(local, "w") as f:
        f.write(iso(utcnow()))
    robust_drive_copy(local, remote)
    os.remove(remote)
    os.remove(local)
    ensure_disk_free("preflight")
    checks = []
    for model in MODELS:
        cand = [r for year, P in units[:40] for m in wanted_months(year) for r in P["months"].get((year, m), [])
                if r["model"] == model and r["id"] not in SKIP_IMAGES]
        if cand:
            checks.append(cross_check(cand[len(cand) // 2]))
    if not checks:
        raise PipelineStop("PREFLIGHT FAILED: no image could be compared with Earth Engine's own reading.")
    log_events("cross_check.jsonl", checks)
    log.info("PREFLIGHT OK (%d collection(s) checked)", len(checks))


def run_pipeline():
    t0 = time.time()
    TIMES.clear()
    TOTALS.clear()
    LAST.clear()
    log.info("=" * 78)
    log.info("  OpenET %s, monthly, 30 m, on tiles | %s | %d variables | TEST_MODE=%s", OPENET_VERSION,
             "+".join(MODELS), len(VARS), TEST_MODE)
    log.info("  output: %s", DRIVE_OUT)
    log.info("=" * 78)
    results, stopped, prog, tiles = [], None, None, []
    try:
        for d in (RAW_DIR, NC_DIR):              # scratch left by an earlier run is never used
            shutil.rmtree(d, ignore_errors=True)
            os.makedirs(d, exist_ok=True)
        ee_login()
        try:
            ee.data.setDefaultWorkloadTag("openet-mask-read")     # lets the compute use be found in Cloud Monitoring
        except Exception:
            pass
        ensure_disk_free("start")
        tiles = build_tiles(get_manifest())
        job = job_months(refresh=True)
        if not job:
            raise PipelineStop(f"OpenET has no month from START_MONTH ({START_MONTH}) to STOP_MONTH ({STOP_MONTH}); "
                               f"its record over the aquifer runs from {ym(*min(AVAILABLE))} to {ym(*max(AVAILABLE))}."
                               if AVAILABLE else "No OpenET image over the aquifer was found.")
        if TEST_MODE:
            log.info("Rehearsal: months %s of %d (START_MONTH and STOP_MONTH are not used)",
                     span([m for _, m in job]), TEST_YEAR)
        else:
            log.info("Months of this run: %s to %s (%d months; OpenET's record runs from %s to %s)", ym(*job[0]),
                     ym(*job[-1]), len(job), ym(*min(AVAILABLE)), ym(*max(AVAILABLE)))
        ALL_TILES[:] = tiles
        log.info("%d tiles hold aquifer pixels: %s pixels of 30 m in all (UTM zones %s)", len(tiles),
                 f"{sum(P['F']['n_roi'] for P in tiles):,}", ", ".join(sorted({P["key"][1:3] for P in tiles})))
        if ONLY_TILES:
            wanted = set(ONLY_TILES)
            absent = wanted - {P["key"] for P in tiles}
            if absent:
                log.warning("ONLY_TILES not available: %s", sorted(absent))
            tiles = [P for P in tiles if P["key"] in wanted]
        if TEST_MODE and tiles:
            n_pick = min(TEST_TILES, len(tiles))
            tiles = [tiles[int((i + 0.5) * len(tiles) / n_pick)] for i in range(n_pick)]
        units = [(year, P) for year in year_list() for P in tiles if wanted_months(year)]
        if not units:
            raise PipelineStop("Nothing to process - check MODELS, the months and ONLY_TILES.")
        n_scenes = lambda year, P: sum(len(P["months"].get((year, m), [])) for m in wanted_months(year))
        prog = Progress(sum(n_scenes(y, P) for y, P in units))
        ledger = load_ledger()
        on_drive = sum(os.path.exists(os.path.join(unit_dir(P["key"], y), unit_name(P["key"], y))) for y, P in units)
        log.info("%d yearly files to have (%d tiles, %s to %s, %d image pieces); %d of them are on Drive already",
                 len(units), len(tiles), ym(*job[0]), ym(*job[-1]), prog.total, on_drive)
        preflight(units)
        skipped = 0
        for k, (year, P) in enumerate(units, 1):
            plan = unit_plan(P, year, ledger)
            if plan["action"] == "skip":
                skipped += 1
                prog.add(n_scenes(year, P), resumed=True)
                continue
            if skipped:
                log.info("%d file(s) already complete on Drive - skipped", skipped)
                skipped = 0
            log.info("-" * 78)
            note = {"new": "", "append": f", added to the months {span(plan['have'])} already in the file",
                    "merge": f", written into a new file with the months {span(plan['have'])} copied from the one on "
                             "Drive"}[plan["action"]]
            log.info("FILE %d/%d  tile %s, %d  (%s aquifer pixels, months %s%s)", k, len(units), P["key"], year,
                     f"{P['F']['n_roi']:,}", span(plan["todo"]), note)
            prog.add(sum(len(P["months"].get((year, m), [])) for m in wanted_months(year) if m in plan["have"]),
                     resumed=True)               # the months of this file that are on Drive already
            results.append(process_unit(P, year, plan, prog))
        if skipped:
            log.info("%d file(s) already complete on Drive - skipped", skipped)
    except PipelineStop as e:
        stopped = str(e)
    except KeyboardInterrupt:
        stopped = "Stopped with the stop button."

    log.info("=" * 78)
    if stopped is not None:
        where = (f"\n  The last file that reached Drive is {LAST['file']}; the next run continues with the one after it."
                 if LAST.get("file") else "\n  No file was finished in this run; the next run starts where this one did.")
        print("\n" + "=" * 78 + f"\n  STOPPED (nothing is lost, re-run to resume)\n  {stopped}{where}\n  Give Drive a "
              "minute to finish uploading before you disconnect the runtime.\n" + "=" * 78 + "\n")
    log.info("  this run: %d yearly file(s) written (%s), %s downloaded in %d blocks, %.1f min", TOTALS.get("files", 0),
             human(TOTALS.get("file_bytes", 0)), human(TOTALS.get("bytes", 0)), TOTALS.get("blocks", 0),
             (time.time() - t0) / 60)
    if TOTALS.get("mask_blocks"):
        log.info("  computed mask reads: %d blocks, %.0f s in all (this is the part that uses Earth Engine compute quota)",
                 TOTALS["mask_blocks"], TIMES.get("mask read (computed)", 0.0))
    if TIMES:
        total_t = sum(TIMES.values())
        log.info("  time by stage: %s", ", ".join(f"{k} {v:.0f} s ({100 * v / total_t:.0f} %)"
                                                   for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1])))
    fresh = [r for r in results if r.get("header_bytes") and r.get("scenes")]
    if TEST_MODE and prog is not None and prog.run and fresh and not TEST_FRAME_PX:
        whole = range_months()
        if whole:
            wanted = set(whole)
            n_all = sum(len(v) for P in ALL_TILES for k, v in P["months"].items() if k in wanted)
            per_scene = (TIMES.get("download", 0.0) + TIMES.get("place + write", 0.0)) / prog.run
            per_px_month = sum(r["size"] - r["header_bytes"] for r in fresh) / sum(r["n_roi"] * r["new_months"] for r in fresh)
            with open(os.path.join(DRIVE_OUT, FRAMES_NAME)) as f:
                all_px = sum(v["n_roi"] for v in json.load(f)["frames"].values() if v)
            n_months = len(whole)
            log.info("  PROJECTION for the months %s to %s (from this rehearsal, rough; summer months are clearer "
                     "than average):", ym(*whole[0]), ym(*whole[-1]))
            log.info("     %d image pieces on %d tiles", n_all, len(ALL_TILES))
            log.info("     ~%.0f h of runtime: %.1f s per image piece", n_all * per_scene / 3600, per_scene)
            log.info("     ~%.1f TB to download: %s per image piece", n_all * TOTALS.get("bytes", 0) / prog.run / 1024 ** 4,
                     human(TOTALS.get("bytes", 0) / prog.run))
            log.info("     ~%.1f TB on Drive: %.1f bytes per aquifer pixel and month, %s pixels x %d months",
                     all_px * n_months * per_px_month / 1024 ** 4, per_px_month, f"{all_px:,}", n_months)
    log.info("=" * 78)
    return results


RESULTS = run_pipeline()

In [ ]:
# ═══════════════ CELL 10: what is on Drive ═══════════════
AUDIT_OPEN_ALL = TEST_MODE     # True -> open and audit every file (slow for a full output folder);
                               # False -> compare each file's size with the note taken when it was uploaded


def audit_all():
    ledger, by_year = load_ledger(), {}
    for p in glob.glob(os.path.join(DRIVE_OUT, "[0-9][0-9][0-9][0-9]", "z*e*n*", f"{PRODUCT_TAG}_*_{ROI_LABEL}.nc")):
        key, year = os.path.basename(os.path.dirname(p)), os.path.basename(os.path.dirname(os.path.dirname(p)))
        if TILE_RE.fullmatch(key) and os.path.basename(p) == unit_name(key, year):
            by_year.setdefault(int(year), []).append((key, p))
    print(f"{DRIVE_OUT}\n")
    print(f"{'year':4s} {'tiles':>5s} {'with all 12 months':>18s} {'size':>10s}  months held (number of tiles)")
    total, n_files, bad, notes = 0, 0, 0, []
    for year in sorted(by_year):
        size, whole, held = 0, 0, {}
        for key, p in sorted(by_year[year]):
            name = os.path.basename(p)
            led, sz = ledger.get(name), os.path.getsize(p)
            size += sz
            if AUDIT_OPEN_ALL or led is None or led.get("size") != sz:
                ok, problems, info = audit_nc(p)
                months = info.get("months", [])
                if not ok:
                    bad += 1
                    notes.append(f"{year} {key}: " + "; ".join(problems))
                elif led is None or led.get("size") != sz:
                    notes.append(f"{year} {key}: not as noted at upload, but it passes the audit")
            else:
                months = [int(m) for m in led.get("months", [])]
            whole += months == list(range(1, 13))
            held[span(months)] = held.get(span(months), 0) + 1
        n_files += len(by_year[year])
        total += size
        print(f"{year:4d} {len(by_year[year]):5d} {whole:18d} {human(size):>10s}  "
              + ", ".join(f"{s or 'none'} ({n})" for s, n in sorted(held.items(), key=lambda kv: -kv[1])))
    print(f"\n{n_files} files, {human(total)} | {bad} file(s) with problems")
    for note in notes:
        print("  " + note)
    for name in (LOG_FAILED, LOG_SKIPPED):
        path, seen = os.path.join(DRIVE_OUT, name), set()
        if os.path.exists(path):
            for line in open(path):
                try:
                    seen.add(json.loads(line).get("image"))
                except ValueError:
                    pass
        print(f"{name}: {len(seen)} images")


audit_all()

# Where the files are: <output folder>/<year>/<tile>/<file>.nc, one file per tile and year.
#
# Reading them. Values are whole millimetres per month as 16-bit integers; xarray turns the fill value
# into NaN by itself:
#   import glob, xarray as xr
#   root  = "/content/drive/MyDrive/MSUGWB/OPENET_v2_1_monthly_30m_tiles"
#   files = sorted(glob.glob(f"{root}/*/z14e02n29/*.nc"))                          # every year of one tile
#   ds = xr.open_mfdataset(files, combine="nested", concat_dim="time", data_vars="minimal", coords="minimal",
#                          compat="override", chunks={"time": 1, "y": 2048, "x": 2048})     # lazily
#   et = ds.et_ensemble_mad                        # mm per month
# A tile has the same name, x and y as in the Landsat index files, so the two can be used together
# pixel for pixel without any regridding.